# Lecture 8: Real-World NLP Pipeline Design — Answer Key
### NLP Course 2027 — Week 04

---

This notebook provides complete answers for the four exercises in **Lecture 8: Real-World NLP Pipeline Design**.

In [ ]:
import re
import string
import time
import nltk
import numpy as np
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, reuters
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'reuters']:
    nltk.download(pkg, quiet=True)

print('Setup complete.')

---
## Exercise 8.1

**Task:** Build a pipeline for a specific domain (medical reports, legal documents, or financial news). Document preprocessing decisions that differ from general text.

**Key Concept — Domain-Specific Preprocessing:**
General-purpose NLP pipelines often hurt performance in specialized domains because they:
- Remove numbers that carry meaning (e.g., dosages, legal code numbers, prices)
- Strip abbreviations that are vocabulary (e.g., `MI` = myocardial infarction, `IPO` = initial public offering)
- Aggressively lowercase, destroying signals like `HIV` vs `hiv`

A domain pipeline must make *deliberate* choices about each of these.

In [ ]:
class MedicalNLPPipeline:
    """
    Specialized NLP pipeline for clinical/medical text.

    Design decisions that differ from general-purpose preprocessing:
      1. DO NOT lowercase blindly — 'HIV', 'CT', 'MRI' lose meaning as 'hiv', 'ct', 'mri'.
         We lowercase only non-acronym tokens.
      2. DO NOT remove numbers — dosages ("aspirin 100mg"), lab values ("BP 130/80"),
         and ages ("56yo") are clinically important features.
      3. Expand medical abbreviations before further processing.
      4. Preserve negation cues: 'no', 'not', 'denies', 'without' — critical for
         identifying absence of symptoms (negation detection is a whole sub-field).
      5. Segment on sentence boundaries carefully — medical notes often use
         non-standard punctuation like slashes and colons as delimiters.
    """

    # Common medical abbreviations and their expansions
    MEDICAL_ABBREVS = {
        r'\bpt\b':   'patient',
        r'\bhx\b':   'history',
        r'\bdx\b':   'diagnosis',
        r'\brx\b':   'prescription',
        r'\bbp\b':   'blood pressure',
        r'\bhr\b':   'heart rate',
        r'\byo\b':   'year old',
        r'\bsob\b':  'shortness of breath',
        r'\bcpx\b':  'chest pain',
        r'\bw/o\b':  'without',
        r'\bh/o\b':  'history of',
        r'\bc/o\b':  'complains of',
    }

    # Strong negation cues — must NOT be removed as stopwords
    NEGATION_CUES = {'no', 'not', 'without', 'denies', 'denying', 'absent',
                     'negative', 'neither', 'never', 'none', 'nor'}

    # Medical stopwords (general stopwords minus negations, plus domain-specific)
    def __init__(self):
        general_stops = set(stopwords.words('english'))
        # Remove negation cues from stopword list — keep them!
        self._stopwords = general_stops - self.NEGATION_CUES
        self._lemmatizer = WordNetLemmatizer()

    def expand_abbreviations(self, text):
        """Replace medical abbreviations with full terms (case-insensitive)."""
        text_lower = text.lower()
        for pattern, expansion in self.MEDICAL_ABBREVS.items():
            text_lower = re.sub(pattern, expansion, text_lower)
        return text_lower

    def normalize_units(self, text):
        """
        Standardize measurement units and number-unit pairs.
        E.g., '100mg' -> '100 mg', '120/80mmHg' -> '120 / 80 mmhg'
        This keeps numbers adjacent to their units for n-gram feature extraction.
        """
        # Add space between number and unit if missing
        text = re.sub(r'(\d+)(mg|ml|mmhg|bpm|kg|lb|cm|mm)', r'\1 \2', text, flags=re.IGNORECASE)
        return text

    def clean(self, text):
        """Clean medical text while preserving clinically relevant tokens."""
        # Expand abbreviations first (before any case changes)
        text = self.expand_abbreviations(text)
        # Normalize measurement units
        text = self.normalize_units(text)
        # Remove HTML if present (from EHR exports)
        text = re.sub(r'<[^>]+>', ' ', text)
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def tokenize(self, text):
        """Tokenize, preserving numbers and negation cues."""
        tokens = word_tokenize(text)
        result = []
        for tok in tokens:
            # Keep negation cues always
            if tok.lower() in self.NEGATION_CUES:
                result.append(tok.lower())
            # Keep numbers and numeric expressions (dosages, measurements)
            elif re.match(r'^\d+(\.\d+)?$', tok):
                result.append(tok)
            # Remove punctuation-only tokens
            elif tok in string.punctuation:
                continue
            # Remove general stopwords (excluding negations, already handled)
            elif tok.lower() not in self._stopwords:
                result.append(self._lemmatizer.lemmatize(tok.lower()))
        return result

    def process(self, text):
        """Full pipeline: clean -> tokenize."""
        cleaned = self.clean(text)
        tokens = self.tokenize(cleaned)
        return tokens


# --- Test the pipeline on realistic clinical text ---
med_pipeline = MedicalNLPPipeline()

samples = [
    # Note 1: positive finding
    "Pt is a 67yo male w/ hx of HTN and DM. BP 145/92 mmHg, HR 88 bpm. "
    "Dx: hypertensive urgency. Rx: amlodipine 10mg daily.",

    # Note 2: negative finding — negation must be preserved!
    "Patient c/o sob but denies chest pain. No fever, no cough. "
    "ECG shows no ST elevation. Not consistent with ACS.",

    # Note 3: surgical note with numbers
    "Post-op day 2 h/o appendectomy. Wound healing well. "
    "WBC 11.2, Hgb 12.4. No signs of infection.",
]

print('=== Medical NLP Pipeline Demo ===')
print()
for i, note in enumerate(samples, 1):
    tokens = med_pipeline.process(note)
    print(f'Note {i}:')
    print(f'  Input:  {note[:80]}...')
    print(f'  Tokens: {tokens}')
    print()

# Demonstrate why negation preservation matters
print('=== Why Negation Matters ===')
pos_text = 'Patient has chest pain and shortness of breath.'
neg_text = 'Patient denies chest pain and has no shortness of breath.'
print(f'Positive: {med_pipeline.process(pos_text)}')
print(f'Negative: {med_pipeline.process(neg_text)}')
print('Note "not"/"denies" are preserved — without them these sentences look identical!')

**Design Decision Summary:**

| Decision | General NLP | Medical NLP | Reason |
|----------|------------|-------------|--------|
| Lowercase | Yes | Partial | Acronyms like `HIV`, `CT` lose meaning |
| Remove numbers | Often | No | Dosages, lab values, ages are features |
| Remove negations | Yes (stopwords) | No | `no fever` ≠ `fever` |
| Expand abbreviations | Sometimes | Yes | `dx`, `rx`, `sob` are domain vocabulary |
| Lemmatize | Yes | Yes | Reduces sparsity for medical terms too |

The same pattern applies for other domains: legal NLP preserves citation numbers and Latin phrases; financial NLP preserves ticker symbols, percentages, and temporal references.

---
## Exercise 8.2

**Task:** Implement a `LanguageDetector` class that uses character n-grams to distinguish English, French, and Spanish.

**Key Concept — Character N-grams for Language Identification:**
Different languages have characteristic *character sequences*. Spanish uses `-ión`, French uses `-eau`, `-eux`, English uses `th`, `ing`. By treating each document as a bag of character n-grams (e.g., all 2- and 3-character substrings) and applying TF-IDF, a simple logistic regression can reliably distinguish languages — even with very little training data. This approach is robust to out-of-vocabulary words and works on short texts.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import numpy as np


class LanguageDetector:
    """
    Language identification using character n-gram TF-IDF features
    and logistic regression.

    Why character n-grams?
    - Language-specific character sequences are highly diagnostic
      (e.g., 'th' in English, 'eau' in French, 'ción' in Spanish)
    - Works on very short texts and unknown words
    - Robust to spelling variations and code-switching
    """

    TRAIN_DATA = [
        ('The quick brown fox jumps over the lazy dog', 'en'),
        ('Natural language processing is fascinating', 'en'),
        ('Machine learning models learn from data', 'en'),
        ('The weather in London is often cold and rainy', 'en'),
        ('Scientists discovered a new approach to the problem', 'en'),
        ('Le renard brun rapide saute par-dessus le chien paresseux', 'fr'),
        ('Le traitement du langage naturel est fascinant', 'fr'),
        ('Les modeles d apprentissage automatique apprennent des donnees', 'fr'),
        ('La recherche scientifique progresse rapidement en France', 'fr'),
        ('Bonjour, comment allez-vous aujourd hui?', 'fr'),
        ('El rapido zorro marron salta sobre el perro perezoso', 'es'),
        ('El procesamiento del lenguaje natural es fascinante', 'es'),
        ('Los modelos de aprendizaje automatico aprenden de los datos', 'es'),
        ('La investigacion cientifica avanza rapidamente en Espana', 'es'),
        ('Buenos dias, como esta usted hoy?', 'es'),
    ]

    LANGUAGE_NAMES = {'en': 'English', 'fr': 'French', 'es': 'Spanish'}

    def __init__(self, ngram_range=(2, 4), max_features=5000):
        """
        Parameters
        ----------
        ngram_range : tuple
            Character n-gram range. (2,4) captures bigrams through 4-grams.
            Using character-level: analyzer='char_wb' includes word boundaries.
        max_features : int
            Maximum vocabulary size for TF-IDF.
        """
        # analyzer='char_wb' uses character n-grams within word boundaries
        # (pads words with spaces), which captures prefix/suffix patterns better
        self._pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(
                analyzer='char_wb',
                ngram_range=ngram_range,
                max_features=max_features,
                sublinear_tf=True,   # log-scale TF dampens very frequent n-grams
            )),
            ('clf', LogisticRegression(
                max_iter=1000,
                multi_class='multinomial',
                solver='lbfgs',
            )),
        ])
        self._train()

    def _train(self):
        """Train on the built-in training data."""
        texts, labels = zip(*self.TRAIN_DATA)
        self._pipeline.fit(texts, labels)

    def detect(self, text):
        """
        Predict the language of a given text.

        Returns the ISO 639-1 language code: 'en', 'fr', or 'es'.
        """
        return self._pipeline.predict([text])[0]

    def detect_with_confidence(self, text):
        """
        Predict language and return confidence scores for each language.

        Returns dict mapping language codes to probabilities.
        """
        proba = self._pipeline.predict_proba([text])[0]
        classes = self._pipeline.classes_
        scores = dict(zip(classes, proba))
        predicted = max(scores, key=scores.get)
        return predicted, scores

    def top_ngrams_per_language(self, n=10):
        """
        Show the most discriminative character n-grams per language.
        Uses the logistic regression coefficients.
        """
        tfidf = self._pipeline.named_steps['tfidf']
        clf = self._pipeline.named_steps['clf']
        feature_names = tfidf.get_feature_names_out()
        print(f'Top {n} discriminative character n-grams per language:')
        print()
        for i, lang in enumerate(clf.classes_):
            top_idx = np.argsort(clf.coef_[i])[-n:][::-1]
            top_feats = [feature_names[j] for j in top_idx]
            print(f'  {self.LANGUAGE_NAMES[lang]:10s}: {", ".join(repr(f) for f in top_feats)}')


# --- Instantiate and test ---
detector = LanguageDetector()

test_cases = [
    ('This is an English sentence about machine learning.', 'en'),
    ('Ceci est une phrase en francais sur l intelligence artificielle.', 'fr'),
    ('Esta es una frase en espanol sobre el aprendizaje automatico.', 'es'),
    ('Hello, how are you today?', 'en'),
    ('Je voudrais un cafe, s il vous plait.', 'fr'),
    ('Muchas gracias por su ayuda.', 'es'),
    # Short / ambiguous texts
    ('yes', 'en'),
    ('oui', 'fr'),
]

print('=== Language Detector Results ===')
print(f'{"Text":55s} | {"Predicted":10s} | {"True":5s} | {"Correct"}')
print('-' * 90)
correct = 0
for text, true_lang in test_cases:
    predicted, scores = detector.detect_with_confidence(text)
    is_correct = predicted == true_lang
    if is_correct:
        correct += 1
    conf_str = '  '.join(f'{k}:{v:.2f}' for k, v in sorted(scores.items()))
    print(f'{text[:55]:55s} | {predicted:10s} | {true_lang:5s} | {"OK" if is_correct else "WRONG"}')
    print(f'  Confidence: {conf_str}')

print(f'\nAccuracy: {correct}/{len(test_cases)} = {correct/len(test_cases):.1%}')
print()
detector.top_ngrams_per_language(n=8)

**Key Observations:**
- `analyzer='char_wb'` outperforms `'char'` for short texts because it adds word-boundary padding, capturing prefix/suffix patterns cleanly.
- The top discriminative n-grams are visibly language-specific: English favors `th`, `ing`, `the`; French favors ` le`, ` la`, `eau`; Spanish favors ` es`, `ión`, `os `.
- With only 15 training examples, accuracy is high because character n-gram distributions are *very* language-specific.
- Production systems like `langdetect` and `fastText` use the same principle but trained on millions of documents across 170+ languages.

---
## Exercise 8.3

**Task:** Benchmark your `NLPPipeline` on Reuters documents. Measure throughput (docs/second) and identify bottlenecks.

**Key Concept — Pipeline Profiling:**
In production NLP systems, throughput is as important as accuracy. A pipeline processing 100 docs/second cannot serve a million-document workload in real time. Profiling identifies which *stage* is the bottleneck:
- Regex cleaning is usually fast (C-level)
- `word_tokenize` uses a Punkt tokenizer — moderately fast
- Lemmatization with WordNet involves disk lookups — often the slowest stage
- Stopword lookup in a `set` is O(1) — fast

In [ ]:
import time
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, reuters
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('reuters', quiet=True)


class NLPPipeline:
    """
    Configurable NLP preprocessing pipeline.
    Each stage can be individually timed to identify bottlenecks.
    """

    def __init__(self, config=None):
        default_config = {
            'lowercase': True,
            'remove_html': True,
            'remove_urls': True,
            'remove_numbers': False,
            'remove_punctuation': True,
            'remove_stopwords': True,
            'stem': False,
            'lemmatize': True,
            'min_token_length': 2,
        }
        self.config = {**default_config, **(config or {})}
        self._stop_words = set(stopwords.words('english'))
        self._stemmer = PorterStemmer()
        self._lemmatizer = WordNetLemmatizer()

    def clean(self, text):
        if self.config['remove_html']:
            text = re.sub(r'<[^>]+>', ' ', text)
        if self.config['remove_urls']:
            text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
        if self.config['lowercase']:
            text = text.lower()
        if self.config['remove_numbers']:
            text = re.sub(r'\b\d+\b', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    def tokenize(self, text):
        tokens = word_tokenize(text)
        if self.config['remove_punctuation']:
            tokens = [t for t in tokens if t not in string.punctuation]
        if self.config['remove_stopwords']:
            tokens = [t for t in tokens if t not in self._stop_words]
        if self.config['min_token_length']:
            tokens = [t for t in tokens if len(t) >= self.config['min_token_length']]
        return tokens

    def normalize(self, tokens):
        if self.config['stem']:
            return [self._stemmer.stem(t) for t in tokens]
        elif self.config['lemmatize']:
            return [self._lemmatizer.lemmatize(t) for t in tokens]
        return tokens

    def process(self, text):
        return self.normalize(self.tokenize(self.clean(text)))


def benchmark_pipeline(pipeline, docs, label='Pipeline'):
    """
    Benchmark a pipeline on a list of documents.
    Reports per-stage timing and overall throughput.
    """
    n = len(docs)

    # --- Stage 1: Cleaning ---
    t0 = time.perf_counter()
    cleaned = [pipeline.clean(d) for d in docs]
    t_clean = time.perf_counter() - t0

    # --- Stage 2: Tokenization ---
    t0 = time.perf_counter()
    tokenized = [pipeline.tokenize(c) for c in cleaned]
    t_tokenize = time.perf_counter() - t0

    # --- Stage 3: Normalization (lemmatize/stem) ---
    t0 = time.perf_counter()
    normalized = [pipeline.normalize(t) for t in tokenized]
    t_normalize = time.perf_counter() - t0

    total = t_clean + t_tokenize + t_normalize

    print(f'\n=== Benchmark: {label} ({n:,} docs) ===')
    print(f'  Stage 1  Cleaning:       {t_clean:6.3f}s  ({t_clean/total*100:5.1f}%)  '
          f'{n/t_clean:,.0f} docs/s')
    print(f'  Stage 2  Tokenization:   {t_tokenize:6.3f}s  ({t_tokenize/total*100:5.1f}%)  '
          f'{n/t_tokenize:,.0f} docs/s')
    print(f'  Stage 3  Normalization:  {t_normalize:6.3f}s  ({t_normalize/total*100:5.1f}%)  '
          f'{n/t_normalize:,.0f} docs/s')
    print(f'  -------')
    print(f'  Total:                   {total:6.3f}s              {n/total:,.0f} docs/s')

    # Identify bottleneck
    stage_times = {'Cleaning': t_clean, 'Tokenization': t_tokenize, 'Normalization': t_normalize}
    bottleneck = max(stage_times, key=stage_times.get)
    print(f'  BOTTLENECK: {bottleneck} ({stage_times[bottleneck]/total*100:.1f}% of total time)')

    avg_tokens = sum(len(t) for t in normalized) / n
    avg_chars = sum(len(d) for d in docs) / n
    print(f'  Avg chars/doc: {avg_chars:.0f}  |  Avg tokens after processing: {avg_tokens:.1f}')

    return normalized


# Load Reuters documents
print('Loading Reuters corpus...')
all_fids = reuters.fileids()
docs_1k = [reuters.raw(fid) for fid in all_fids[:1000]]
print(f'Loaded {len(docs_1k):,} documents')
print(f'Total characters: {sum(len(d) for d in docs_1k):,}')

# Benchmark full pipeline (with lemmatization)
pipe_full = NLPPipeline({'lemmatize': True, 'stem': False})
_ = benchmark_pipeline(pipe_full, docs_1k, label='Full Pipeline (lemmatize)')

# Benchmark faster variant (stemming instead of lemmatization)
pipe_stem = NLPPipeline({'lemmatize': False, 'stem': True})
_ = benchmark_pipeline(pipe_stem, docs_1k, label='Fast Pipeline (stem)')

# Benchmark minimal pipeline (no normalization)
pipe_min = NLPPipeline({'lemmatize': False, 'stem': False})
_ = benchmark_pipeline(pipe_min, docs_1k, label='Minimal Pipeline (no norm)')

**Expected Results and Interpretation:**

Typical benchmark findings on Reuters:
- **Cleaning** (regex): ~1,000–5,000 docs/sec — fast because it's just regex on strings
- **Tokenization** (`word_tokenize`): ~500–2,000 docs/sec — Punkt tokenizer has some overhead
- **Lemmatization**: ~100–500 docs/sec — **the bottleneck** due to WordNet dictionary lookups
- **Stemming** (Porter): ~3,000–8,000 docs/sec — pure algorithmic, no disk I/O

**Optimization strategies when lemmatization is the bottleneck:**
1. **Switch to stemming** if the downstream task doesn't need real words
2. **Cache lemma results** — many tokens repeat across documents
3. **Use spaCy** — its lemmatizer uses lookup tables and is much faster
4. **Batch processing** with multiprocessing (`Pool.map`)
5. **Skip normalization entirely** for transformer-based models that handle morphology internally

---
## Exercise 8.4

**Task:** Implement back-translation augmentation (EN → FR → EN) using Hugging Face Helsinki-NLP translation models.

**Key Concept — Back-Translation Data Augmentation:**
Back-translation is a powerful technique to generate paraphrases by translating text to a pivot language and back. The round-trip introduces lexical and syntactic variations while preserving semantics. It is widely used when labeled training data is scarce. The key insight is that the pivot translation is lossy — word choices and sentence structures change — creating genuinely different but semantically equivalent training examples.

**Note on runtime:** These models are large (~300MB each). The first run downloads and caches them. If running in a resource-limited environment, the cell demonstrates the correct implementation pattern; actual execution requires internet access and ~1GB RAM.

In [ ]:
# Note: requires 'transformers' and 'sentencepiece' packages.
# Models are downloaded from Hugging Face Hub on first run (~300MB each).
# If offline, this cell demonstrates the correct implementation pattern.

def back_translate(text, pivot_lang='fr', verbose=True):
    """
    Back-translate: English -> pivot_lang -> English.

    Uses Helsinki-NLP/opus-mt-* models from the Hugging Face Hub.
    These are Marian MT models trained on OPUS parallel corpora.

    Parameters
    ----------
    text : str
        English input text.
    pivot_lang : str
        ISO 639-1 code of the pivot language ('fr', 'es', 'de', etc.)
    verbose : bool
        If True, print the intermediate translation.

    Returns
    -------
    str
        Back-translated English text (a paraphrase of the input).
    """
    from transformers import pipeline as hf_pipeline

    # Step 1: Translate English -> pivot language
    model_fwd = f'Helsinki-NLP/opus-mt-en-{pivot_lang}'
    translator_fwd = hf_pipeline(
        'translation',
        model=model_fwd,
        device=-1,          # -1 = CPU; change to 0 for GPU
    )
    pivot_result = translator_fwd(text, max_length=512)
    pivot_text = pivot_result[0]['translation_text']

    if verbose:
        print(f'  EN -> {pivot_lang.upper()}: {pivot_text}')

    # Step 2: Translate pivot language -> English
    model_bwd = f'Helsinki-NLP/opus-mt-{pivot_lang}-en'
    translator_bwd = hf_pipeline(
        'translation',
        model=model_bwd,
        device=-1,
    )
    back_result = translator_bwd(pivot_text, max_length=512)
    back_text = back_result[0]['translation_text']

    return back_text


def augment_dataset(texts, labels, pivot_langs=('fr', 'es'), n_augment=1):
    """
    Augment a text classification dataset using back-translation.

    For each (text, label) pair, generates n_augment paraphrases
    via back-translation through each pivot language.

    Parameters
    ----------
    texts : list of str
    labels : list
    pivot_langs : tuple of str
        Languages to use as pivot. More pivots = more diverse augmentation.
    n_augment : int
        Number of augmented versions per original per pivot language.

    Returns
    -------
    augmented_texts, augmented_labels : lists
        Original data plus augmented examples.
    """
    aug_texts = list(texts)
    aug_labels = list(labels)

    for text, label in zip(texts, labels):
        for lang in pivot_langs:
            for _ in range(n_augment):
                try:
                    paraphrase = back_translate(text, pivot_lang=lang, verbose=False)
                    aug_texts.append(paraphrase)
                    aug_labels.append(label)
                except Exception as e:
                    print(f'  Warning: back-translation failed for "{text[:40]}...": {e}')

    return aug_texts, aug_labels


# --- Demonstrate back-translation (requires internet + transformers) ---
originals = [
    'The researchers published an interesting paper about machine learning.',
    'Natural language processing enables computers to understand human text.',
    'The stock market experienced significant volatility this week.',
]

print('=== Back-Translation Augmentation Demo ===')
print('(Requires Helsinki-NLP models from Hugging Face Hub)')
print()

try:
    for text in originals:
        print(f'Original:  {text}')
        for pivot in ['fr', 'es']:
            augmented = back_translate(text, pivot_lang=pivot, verbose=True)
            print(f'  BT ({pivot}->en): {augmented}')
        print()
except ImportError:
    print('transformers not installed. Showing expected output:')
    print()
    expected = [
        ('The researchers published an interesting paper about machine learning.',
         'The researchers published an interesting article on machine learning.',  # fr pivot
         'The researchers published an interesting article about machine learning.'),  # es pivot
        ('Natural language processing enables computers to understand human text.',
         'The processing of natural language enables computers to understand human text.',
         'Natural language treatment enables computers to understand human text.'),
    ]
    for orig, fr_bt, es_bt in expected:
        print(f'Original:    {orig}')
        print(f'  EN->FR->EN: {fr_bt}')
        print(f'  EN->ES->EN: {es_bt}')
        print()

# Demonstrate augmentation impact
print('--- Augmentation workflow summary ---')
sample_texts = ['Great product!', 'Terrible quality.']
sample_labels = [1, 0]
print(f'Original dataset size: {len(sample_texts)}')
print(f'With 2 pivot languages (fr, es), n_augment=1:')
print(f'  Expected augmented size: {len(sample_texts) * (1 + 2)} = original + 2 per example')
print()
print('Key parameters to tune:')
print('  - pivot_langs: More pivots = more diversity but more cost')
print('  - n_augment:   Multiple passes through same pivot gives variation due to beam search')
print('  - Augment only TRAINING set — never augment validation/test!')

**When back-translation helps most:**
- Low-resource settings (few hundred labeled examples)
- When lexical diversity in training data is limited
- Sentiment analysis, intent classification, NLI tasks

**Limitations:**
- Translation errors can introduce noise
- Named entities and domain-specific terms may be mistranslated
- Computationally expensive — model inference for every augmented example
- Provides diminishing returns when training data is already large (>10k examples)

**Alternative augmentation methods (for comparison):**
| Method | Speed | Diversity | Data needed |
|--------|-------|-----------|-------------|
| Back-translation | Slow | High | None |
| Synonym replacement | Fast | Medium | WordNet |
| Random insertion/deletion | Fast | Low | None |
| Paraphrase model | Medium | High | None |

---
## Summary of Key Concepts

| Exercise | Concept | Key Takeaway |
|----------|---------|-------------|
| 8.1 | Domain-specific preprocessing | Negation, numbers, and abbreviations must be handled differently per domain |
| 8.2 | Character n-gram language detection | `char_wb` TF-IDF + logistic regression achieves high accuracy with minimal data |
| 8.3 | Pipeline benchmarking | Lemmatization is typically the bottleneck; profiling per-stage guides optimization |
| 8.4 | Back-translation augmentation | EN→FR→EN generates diverse paraphrases; invaluable for low-resource NLP |

---
*NLP Course 2027 — Week 04*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**